### **Laboratorio 10: Memoria persistente sobre reuniones reales**

El laboratorio es autónomo, puede abrirse sin ejecutar antes el cuaderno teórico. 

- Datos: QMSum Academic/Committee original.
- Unidad experimental: cada pregunta humana de una reunión y los turnos de evidencia anotados. No agregamos preguntas sintéticas ni fingimos respuestas de LLM.

**Objetivos observables:** verificar fuentes, ejecutar dos baselines, comparar casos por ID, probar contratos de escritura, investigar la relación recencia/relevancia y defender limitaciones.

#### **Etapa 0: Contrato de entrada y procedencia**

Antes de medir, identifica el commit fuente, los seis archivos oficiales, las particiones y los hashes. La celda falla si faltan documentos: no existe fallback de juguete.

In [ ]:
from pathlib import Path
import json, sys, tempfile
import pandas as pd

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'Semana10'/'memoria_real.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Usa JupyterLab dentro de /workspace/CC-0F4')
if str(root) not in sys.path:
    sys.path.insert(0,str(root))
from Semana10 import memoria_real as m
provenance = m.verify()
print('Commit de los archivos:', provenance['source_commit'])
display(pd.DataFrame(provenance['files']).T[['meetings','specific_queries','sha256']])


#### **Etapa 1 : Congelar políticas y métricas**

A = últimos **48 turnos** de la reunión, B = **BM25 top-48** sobre todos los turnos guardados en SQLite. Comparar con la misma pregunta y el mismo presupuesto de turnos. No se usa ningún LLM ni se reporta exactitud de resumen.

**Antes de correr**, escribe una predicción sobre qué política tendrá mayor Recall medio y una posible causa de fracaso de cada una. Las predicciones no son resultados.

#### **Formalización**

$$\mathrm{Recall}(q,P)=|G_q\cap S_q^P|/|G_q|,\quad
\mathrm{AnyHit}(q,P)=\mathbb{1}[G_q\cap S_q^P\ne\varnothing].$$

No confundas `n_selected=48` con 48 fragmentos pertinentes. `AnyHit=1` tampoco significa haber localizado toda la discusión.

#### **Etapa 2:  Evaluación en la partición de prueba**

Ejecuta el **test oficial completo** de Academic y Committee con los valores fijados. Se creará una base SQLite persistente y archivos de resultados por consulta. Si no descargaste QMSum antes de clase, prepara primero con `python -m Semana10.memoria_real --preparar` desde Terminal.

In [ ]:
test_summary = m.run('test', limit=48)
print('Reuniones evaluadas:', test_summary['n_meetings'])
print('Registros consulta-política:', test_summary['n_evaluated_query_policy_rows'])
display(pd.DataFrame(test_summary['results']).T)
print('Interpretación: solo recuperación de evidencia, no calidad de respuestas')


#### **Etapa 3 : Comparación pareada sobre las mismas consultas**

Las medias pueden ocultar fallos. Construye una tabla por `meeting` y `query_index`. Muestra casos donde BM25 mejora, empeora o empata frente a la ventana reciente. No elimines casos negativos ni "difíciles".

In [ ]:
rows = [json.loads(x) for x in (m.OUT/'qmsum_test_por_consulta.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]
df = pd.DataFrame(rows)
assert len(df) == test_summary['n_evaluated_query_policy_rows']
pv = df.pivot(index=['meeting','query_index'],columns='policy',values='span_turn_recall')
pv['delta'] = pv['episodic_bm25'] - pv['working_recent']
print('Diferencias positivas:',int((pv['delta']>0).sum()))
print('Diferencias negativas:',int((pv['delta']<0).sum()))
print('Empates:',int((pv['delta']==0).sum()))
display(pv.sort_values('delta',ascending=False).head(8))


#### **Etapa 4 : Auditoría de errores con turnos originales**

Identifica un caso donde BM25 recupere más evidencia, otro donde la ventana reciente recupere más y un empate si existen. La explicación debe identificar la pregunta humana, el índice de los turnos gold y al menos un fragmento original. Si una categoría está vacía, documenta la ausencia en vez de fabricar un caso.

In [ ]:
meetings = dict(m.get_meetings('test'))

def show_case(index, name):
    if index is None:
        print(name, ': no hay casos en esta categoría en el test actual')
        return
    mid, qi = index
    original = meetings[mid]
    question = original['specific_query_list'][int(qi)]
    gold = sorted(m.gold_positions(question, len(original['meeting_transcripts'])))
    print('\nCASO',name,'|',mid,'| consulta',qi)
    print('Pregunta:',question['query'])
    print('Turnos gold:',len(gold),'| primeros',gold[:10])
    for i in gold[:3]:
        turn = original['meeting_transcripts'][i]
        print('  Turno',i,'/',turn.get('speaker'),':',str(turn.get('content',''))[:300])

better = pv[pv['delta']>0]
worse = pv[pv['delta']<0]
equal = pv[pv['delta']==0]
show_case(better.sort_values('delta',ascending=False).index[0] if len(better) else None,'BM25 mejora')
show_case(worse.sort_values('delta').index[0] if len(worse) else None,'BM25 empeora')
show_case(equal.index[0] if len(equal) else None,'Empate')


**Ejercicio obligatorio A.** Describe dos mecanismos observados, no hipotéticos: por qué un turno gold quedó fuera de la ventana final y por qué una pregunta pudo quedar mal servida por BM25 (paráfrasis, palabras no compartidas, discusión dispersa o empate de score). Indica el `meeting`, `query_index` y el texto original.

#### **Etapa 5: La persistencia debe sobrevivir a otra conexión**

Prueba una reunión **real** en un SQLite temporal: guardar, cerrar y volver a abrir. Una segunda escritura idéntica no agrega duplicados. Este ensayo verifica persistencia de almacenamiento, no el rendimiento de recuperación.

In [ ]:
mid, meeting = next(iter(meetings.items()))
with tempfile.TemporaryDirectory(prefix='s10_persistencia_') as tmp:
    dbfile = Path(tmp)/'episodios.sqlite3'
    with m.MemoryStore(dbfile) as db:
        print('Guardar:',db.save_meeting(mid,meeting['meeting_transcripts']))
    with m.MemoryStore(dbfile) as db:
        print('Reingresar:',db.save_meeting(mid,meeting['meeting_transcripts']))
        recovered = db.read(mid)
    print('Turnos originales:',len(meeting['meeting_transcripts']),'| recuperados:',len(recovered))
    assert len(recovered) == len(meeting['meeting_transcripts'])


#### **Etapa 6: Inyectar un conflicto de identidad, conservar datos**

Un error habitual es reemplazar datos silenciosamente si se usa `INSERT OR REPLACE`. Reutilizamos dos reuniones originales distintas bajo **una misma clave artificial**, exclusivamente para probar que la escritura se rechaza. No constituye un caso experimental de QMSum.

In [ ]:
pairs = list(meetings.items())
assert len(pairs)>1
with tempfile.TemporaryDirectory(prefix='s10_conflicto_') as tmp:
    with m.MemoryStore(Path(tmp)/'conflictos.sqlite3') as db:
        db.save_meeting('identidad_en_disputa',pairs[0][1]['meeting_transcripts'])
        original_count=len(db.read('identidad_en_disputa'))
        try:
            db.save_meeting('identidad_en_disputa',pairs[1][1]['meeting_transcripts'])
        except ValueError as e:
            print('Rechazo esperado:',str(e)[:170])
        else:
            raise AssertionError('La base aceptó un reemplazo no autorizado')
        assert len(db.read('identidad_en_disputa')) == original_count


**Ejercicio obligatorio B.** Explica por qué esta protección resuelve un problema de integridad por identificador, pero **no** determina cuál de dos afirmaciones contradictorias de una reunión es verdadera. Propón los campos de auditoría que necesitarías para modelar vigencia y correcciones, sin inventar etiquetas de QMSum.

#### **Etapa 7: Diagnóstico de score léxico y abstención**

El score BM25 más alto puede ser cero o muy pequeño. Eso permite diagnosticar ausencia de coincidencia de términos, pero **no autoriza afirmar que la consulta está sin respuesta**. QMSum no aporta negativos independientes para medir exactitud de abstención.

In [ ]:
lex = df[df['policy']=='episodic_bm25'].copy()
zero = lex[lex['top_bm25_score']==0]
print('Consultas humanas con score léxico máximo igual a cero:',len(zero))
print('Consultas de evaluación:',len(lex))
if len(zero):
    display(zero[['meeting','query_index','top_bm25_score','span_turn_recall']].head(8))
print('No se reporta accuracy de abstención porque faltan negativos etiquetados.')


#### **Etapa 8 : Costos, capacidad y decisiones de implementación**

Ambos sistemas seleccionan hasta 48 turnos, pero el índice BM25 procesa el texto y calcula puntuaciones para las consultas. La ventana reciente cuesta menos procesamiento, aunque puede perder evidencia temprana. Comparar tiempo de CPU o tokens de LLM **no** es parte de este protocolo y necesitaría instrumentación adicional.

**Ablación opcional solo en validación:** ejecuta `m.run('val', limit=16)` y `m.run('val', limit=96)` para observar cobertura frente a presupuesto. La función guarda resultados con sufijos distintos para no sobrescribir el test de 48. No recalibres `test` después de observar su resultado.

#### **Etapa 9 : Exportar reporte de evidencia, no solo capturas**

Todos los archivos numéricos deben derivarse de la ejecución actual y estar identificados por partición, commit fuente y limitaciones. Conserva `qmsum_test_resumen.json` y `qmsum_test_por_consulta.jsonl` si documentas la práctica. No inventes filas ni edites los gold spans.

In [ ]:
summary_path=m.OUT/'qmsum_test_resumen.json'
rows_path=m.OUT/'qmsum_test_por_consulta.jsonl'
assert summary_path.is_file() and rows_path.is_file()
print('Resumen real:',summary_path.relative_to(root))
print('Resultados por consulta:',rows_path.relative_to(root))
print('Datos de origen: QMSum Academic/Committee')
print('Límite fijo de turnos:',test_summary['limit'])


#### **Entrega y defensa (hasta dos páginas)**

Entrega: notebook ejecutado sin errores, resumen JSON y resultados por consulta generados realmente, dos casos justificados con IDs y fragmentos originales, y respuesta a los ejercicios A y B.

**Defensa individual:**

1. Diferencia estado de ejecución, memoria de trabajo y episódica.
2. Justifica un fallo real observado de cada política, o la ausencia de la categoría.
3. Explica qué dato pertenece al recuperador y qué dato pertenece a la evaluación.
4. Demuestra que la persistencia idempotente no sobrescribe episodios.
5. Argumenta por qué vigencia y abstención no están validadas por este experimento.

No utilices este laboratorio para afirmar superioridad de agentes LLM: no se generó ninguna respuesta con un modelo.